# The Shrinkage Question
### An instructor's worked exemplar — BDAT05 · Fundamentals of Predictive Analytics

**Jerald B. Bongalos** · LPU–Laguna, College of Business and Accountancy · Term 1, AY 2026–2027

---

**What this notebook is.** This is the model of the *format* your own notebooks must follow. Every step in it has three parts, always in the same order:

> **💬 Discussion** — a Markdown cell that says *what we are about to do and why*, before any code.
> **▶ Code** — one small, commented cell that does exactly one job.
> **🔎 Reading the output** — a Markdown cell that interprets *what the output actually says*, written after running the cell.

A notebook with code but no reading cells is a script. A notebook with reading cells is **analysis**. You are graded on the second one.

**The data.** We use two Kape Tayo files that belong to *no one's track*: `inventory_movements.csv` (the fact table) with `products.xlsx` and `branches.csv` as lookup tables. Your own slices — branch-month revenue, product-month revenue, workforce, order counts — stay untouched; this exemplar will not do your P-project for you.

**AI-use statement.** Parts of this notebook's code were drafted with an LLM assistant under the course protocol (*The Auditable Prompt*: brief → frame → plan → verify → log). The full log entry appears at the end — Tier 3, disclosed, verified. Yours must too.


---
## Step 0 · The frame — before any code

The canvas comes first, always. Six fields, exactly as in P1:

| Field | This analysis |
|---|---|
| **Business decision** | Each month, the supply-chain lead decides **which products get tighter stock handling** and **which branches get a process audit**. |
| **Decision-maker** | Supply-chain lead (reports to the Operations Manager). |
| **Question** | Where does shrinkage — stock lost to wastage — **concentrate**, by product and by branch, and what does it cost in pesos? |
| **Unit of analysis** | One row = one **inventory movement event** (raw); analysis grain = **product** and **branch** totals. |
| **Data window** | January 2025 – June 2026 (18 months). |
| **Success criterion** | The lead can name the top-3 peso-loss products and the highest-loss branch, with numbers they trust. |

Notice this is a **descriptive** question — *where and how much*, not *what will happen next*. That is deliberate: you earn the right to predict only after you can describe. The closing section says when this question would turn predictive.


---
## Step 1 · Set up the environment

**💬 Discussion.** One cell: imports for the course stack and a printout of versions. The versions are not decoration — your notebook is graded by whether it re-runs top-to-bottom (*Restart & Run all*), and a printed version line records exactly what produced your numbers. We import only what this notebook uses; scikit-learn and statsmodels would be imported at the cell where they first appear — this analysis never needs them.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)   # never let a hidden column hide a problem
print("pandas", pd.__version__, "| numpy", np.__version__)

**🔎 Reading the output.** Two version numbers and nothing else — which is the point. A setup cell that produces errors or warnings is a finding in itself; a silent, versioned setup cell is a clean handshake with the environment. If your versions differ from a classmate's and your numbers ever disagree, this line is where the investigation starts.


---
## Step 2 · Load the data from Google Drive

**💬 Discussion.** We use **Route B** from the course deck — the share-link route, no mount needed. Each Drive file has a `FILE_ID` (the long token in its share link); `pd.read_csv` and `pd.read_excel` read straight from the download URL. Note that one of our three files is an Excel workbook — pandas does not care, the reading pattern is identical.

*If a URL read fails with an HTML-looking error, the file is not link-shared — fall back to Route A (`drive.mount`) exactly as shown in the deck.*

Privacy check before we load: movements, products, branches — **no personal data anywhere in these files**, so the lawful-basis question is light today. Your track may not be so lucky; check before you load, not after.


In [ ]:
def drive_url(file_id):
    """Direct-download URL for a link-shared Google Drive file."""
    return f"https://drive.google.com/uc?export=download&id={file_id}"

FILE_IDS = {
    "movements": "1vqLw8uXdi1yLTAeWWZtKQEgjbBANCLV7",   # inventory_movements.csv
    "products":  "1thppJ6xhPDc961dw_cLAA2utSaoR8iOr",   # products.xlsx
    "branches":  "1QgVoziJ95cLZGkyX1r1i12KJccFopWtB",   # branches.csv
}

mov      = pd.read_csv(drive_url(FILE_IDS["movements"]))
products = pd.read_excel(drive_url(FILE_IDS["products"]))
branches = pd.read_csv(drive_url(FILE_IDS["branches"]))

print("movements:", mov.shape, "| products:", products.shape, "| branches:", branches.shape)

**🔎 Reading the output.** Three tables landed: **1,541 movement events**, **12 products**, **6 branches**. The shapes already teach us the roles — one long fact table, two short lookup tables. That 1,541 is our first honest number: whatever story we tell about shrinkage, it comes from at most 1,541 rows, and (as we are about to discover) far fewer once we isolate wastage.


---
## Step 3 · The handshake — look before you trust

**💬 Discussion.** Before any analysis: `info()` and `head()`. We are looking for three things — row/column counts, **dtypes** (is each column stored as what it claims to be?), and missing values. This is the single habit that separates analysts from people who get surprised in Week 10.


In [ ]:
mov.info()
mov.head()

**🔎 Reading the output.** Row count confirms 1,541, no missing values anywhere — suspiciously tidy, and we stay suspicious. The real find is in the **Dtype column: `QuantityMoved` is stored as text** (shown as `object` or `str` depending on your pandas version) — not a number. A quantity column stored as text cannot be summed, and `.sum()` on it would not crash — it would silently concatenate strings or refuse. `Date` is also text, which is normal for a fresh CSV read but must be fixed before any time analysis. Two dtype repairs are now on the work list *before* we compute anything. This is why the handshake exists: **df.info() found in ten seconds what a wrong total would have hidden for weeks.**


---
## Step 4 · Validate — find out *why* before touching anything

**💬 Discussion.** `QuantityMoved` is text for a reason, and we do not clean what we have not diagnosed. Three checks: what values live in `MovementType` (categories are where inconsistency hides), which exact rows break the number conversion (`pd.to_numeric` with `errors="coerce"` turns unconvertible values into `NaN` so we can *see* them), and whether every key in the fact table exists in its lookup table (orphan check — the Session 3 lesson).


In [ ]:
# 1 · What categories exist?
print(mov["MovementType"].value_counts(), "\n")

# 2 · Which rows refuse to be numbers?
as_num = pd.to_numeric(mov["QuantityMoved"], errors="coerce")
broken = mov[as_num.isna()]
print("rows that fail numeric conversion:", len(broken))
print(broken[["MovementID", "Date", "MovementType", "QuantityMoved"]].head(), "\n")

# 3 · Orphan keys + duplicates
print("orphan ProductIDs:", set(mov["ProductID"]) - set(products["ProductID"]))
print("orphan BranchIDs: ", set(mov["BranchID"])  - set(branches["BranchID"]))
print("duplicate MovementIDs:", mov["MovementID"].duplicated().sum())

# 4 · The sign question — are negatives errors?
print("\nnegative quantities by type:")
print(mov.assign(q=as_num).query("q < 0").groupby("MovementType")["q"].agg(["count", "sum"]))

**🔎 Reading the output.** Four findings, three of them problems and one of them *not*:

1. **Case inconsistency** — `Delivery` (1,451 rows) and `delivery` (25 rows) are the same event wearing two spellings. Any `groupby` on this column would silently split one category into two.
2. **Fourteen rows carry thousands separators** — values like `4,195` — and *commas are why the whole column is text*. Note what else the broken rows share: they are all Deliveries, and all suspiciously large (a normal delivery tops out near 120 units). Recovering the number is easy; whether a 4,000-unit delivery is *real* is a business question, not a pandas question. We will flag, not delete.
3. **No orphans, no duplicates** — every movement points at a real product and a real branch. The join is safe.
4. **Negative quantities are not errors.** They appear *only* on Wastage, Adjustment, and Transfer Out — outbound movement types. That is a **sign convention**: outflows are negative by design. Finding a pattern and recognizing it as *structure* rather than *dirt* is the judgment call validation exists to teach.


---
## Step 5 · Clean — decisions, not deletions

**💬 Discussion.** Each fix below is a *decision with a reason*, and the cell prints shape before and after because **nothing is allowed to change silently**. We fix the case variants, strip the commas and restore the numeric dtype, parse dates, and add an `IsFlagged` column for the fourteen giant deliveries — kept in the data, marked for the supply-chain lead to confirm. Deleting them would be inventing a fact.


In [ ]:
print("before:", mov.shape)

clean = mov.copy()

# Fix 1 — one category, one spelling
clean["MovementType"] = clean["MovementType"].str.strip().str.title()

# Fix 2 — strip thousands separators, restore numeric dtype
clean["QuantityMoved"] = pd.to_numeric(
    clean["QuantityMoved"].astype(str).str.replace(",", "", regex=False)
)

# Fix 3 — real dates
clean["Date"] = pd.to_datetime(clean["Date"])

# Fix 4 — flag (not delete) implausibly large deliveries, pending business confirmation
clean["IsFlagged"] = clean["QuantityMoved"] > 500

print("after: ", clean.shape, "— same rows, nothing dropped")
print(clean["MovementType"].value_counts())
print("flagged rows:", clean["IsFlagged"].sum())
print(clean.dtypes)

**🔎 Reading the output.** Same 1,541 rows in, 1,541 out — the cleaning changed *representations*, never *facts*. `Delivery` is now one category of **1,476** (1,451 + the 25 lowercase strays), `QuantityMoved` is `int64` and can finally do arithmetic, `Date` is `datetime64`, and exactly **14 rows carry the flag**. The cleaning log, as it will appear in a P2 submission:

| # | Issue | Decision | Reason |
|---|---|---|---|
| 1 | `delivery` vs `Delivery` (25 rows) | Normalize to title case | Same real-world event; two spellings would split every groupby |
| 2 | Commas in 14 `QuantityMoved` values | Strip separator, convert to `int64` | Values are recoverable; text dtype blocked all arithmetic |
| 3 | `Date` stored as text | Parse to `datetime64` | Required for any time-based grouping |
| 4 | 14 deliveries of 1,582–4,663 units | **Flag, keep, refer to business** | 10–40× a typical delivery; could be bulk orders or entry errors — deleting would fabricate certainty we do not have |


---
## Step 6 · Transform — build the analysis table

**💬 Discussion.** Now, and only now, the actual question. We isolate **Wastage** events, flip the sign convention into a readable `UnitsLost`, and join both lookup tables — product names and unit costs from the Excel file, branch names from the CSV. Multiplying units lost by `UnitCost` prices the shrinkage in pesos, because the supply-chain lead does not think in units of Ensaymada; they think in money. State your grain out loud before you aggregate: **one row = one wastage event**, rolled up to product totals and branch totals.


In [ ]:
waste = (
    clean.query("MovementType == 'Wastage'")
         .assign(UnitsLost=lambda d: -d["QuantityMoved"])       # flip the sign convention
         .merge(products, on="ProductID", how="left")            # names, category, UnitCost
         .merge(branches[["BranchID", "BranchName"]], on="BranchID", how="left")
         .assign(PesoLoss=lambda d: d["UnitsLost"] * d["UnitCost"])
)
print("wastage events:", len(waste), "| units lost:", waste["UnitsLost"].sum(),
      "| total peso loss:", waste["PesoLoss"].sum())

by_product = (waste.groupby(["ProductName", "Category"], as_index=False)
                    .agg(Events=("UnitsLost", "count"),
                         Units=("UnitsLost", "sum"),
                         PesoLoss=("PesoLoss", "sum"))
                    .sort_values("PesoLoss", ascending=False))
print("\n", by_product.to_string(index=False))

by_branch = (waste.groupby("BranchName", as_index=False)
                   .agg(Events=("UnitsLost", "count"), PesoLoss=("PesoLoss", "sum"))
                   .sort_values("PesoLoss", ascending=False))
print("\n", by_branch.to_string(index=False))

**🔎 Reading the output.** The headline: **23 wastage events, 158 units, ₱11,399** over 18 months. Small numbers — and saying so is part of the analysis. Then two very different concentrations:

- **By product**, the top two losses tell two different stories. The **Tumbler 350ml** leads at **₱3,120** from a *single* 13-unit event — that reads like one breakage or write-off incident, not a pattern. **Tsokolate de Batirol** is second at **₱2,880** across *four* events — that reads like a recurring perishable-handling problem. Same column, opposite interventions: an incident report for one, a process change for the other. **The event count changes the meaning of the peso total** — never report one without the other.
- **By branch**, **Biñan carries ₱4,864** — roughly **43% of all peso shrinkage** — while Los Baños lost ₱54 all period. If the lead can audit one branch this quarter, the data has already chosen it.

And the caution that keeps us honest: with only 23 events, these are *descriptive* facts about the past 18 months, not stable rates. **We counted our rows before trusting our story** — the same discipline your P1 grain feedback demanded.


---
## Step 7 · One chart, decision first

**💬 Discussion.** One figure, because the decision needs one: *which products get tighter handling?* A horizontal bar of peso loss per product, sorted, so the top of the chart is the top of the priority list. Title, labeled axis with units, no legend needed — a chart the supply-chain lead can read without us in the room.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = by_product.sort_values("PesoLoss")           # smallest at bottom → biggest on top
ax.barh(plot_df["ProductName"], plot_df["PesoLoss"], color="#A6192E")
ax.set_title("Wastage cost by product, Jan 2025 – Jun 2026", fontsize=13)
ax.set_xlabel("Peso loss (₱, at unit cost)")
ax.spines[["top", "right"]].set_visible(False)
for y, v in enumerate(plot_df["PesoLoss"]):
    ax.text(v + 40, y, f"₱{v:,.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

**🔎 Reading the output.** The bar lengths make the memo write itself: two products account for **₱6,000 of the ₱11,399 total** — more than half the shrinkage cost sits in two SKUs. Everything below Spanish Latte is noise at this scale. Note what the chart *cannot* say: it hides the event counts, which is exactly why the reading cell above it exists — the figure ranks the losses, the table explains them.

**The figure, captioned for a report (APA 7):**

> **Figure 1**
> *Wastage Cost by Product at Kape Tayo Coffee, January 2025 – June 2026*
> *Note.* Peso loss valued at unit cost; aggregated from 23 wastage events in inventory movement records. No personal data.

And a result sentence in APA style, if this fed a formal write-up:

> Shrinkage cost was concentrated in merchandise and non-coffee items, with the top two products accounting for ₱6,000 of the ₱11,399 total loss (52.6%) recorded over the 18-month window.


---
## Step 8 · References and the audit trail

**References (APA 7).** Software and data are sources; cite them like sources — and verify every entry against the real publication, because LLMs invent references with total confidence:

> Bongalos, J. B. (2026). *Kape Tayo Coffee integrated case dataset* [Unpublished raw data]. LPU–Laguna, College of Business and Accountancy.
>
> Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55
>
> The pandas development team. (2024). *pandas* (Version 2.x) [Computer software]. https://pandas.pydata.org

**The AI-use log entry for this notebook** — the six fields, filled in:

| Field | Entry |
|---|---|
| Date & task | 27 Aug 2026 · Instructor exemplar — shrinkage analysis, inventory movements |
| Tool & conversation link | LLM assistant · *(shared conversation link filed with the course records)* |
| Tier | Tier 3 — model-drafted code under the course protocol |
| Prompted for | Assistant brief · frame · plan · cleaning cell · groupby/merge cell · one chart · APA caption |
| Kept / changed | Kept plan and cleaning approach; rewrote the flag threshold (model proposed deleting outliers — rejected); tightened chart labels |
| Verified by | Row counts before/after every step · category totals recomputed by hand for one branch · Restart & Run all clean |

Read the *Kept / changed* line twice. The model proposed deleting the flagged deliveries; the analyst said no. **That line is where the supervision shows** — a log that says "kept everything" is not a good log, it is a confession that nobody checked.


---
## Closing · What you just watched, and what P2 asks of you

Every step followed the same contract — **Discussion → Code → Reading the output** — and that contract is the deliverable format for every notebook you submit in this course. The checklist to imitate:

1. **The frame came before the code** — six canvas fields, no imports until the question was on paper.
2. **Every code cell did one job** and printed evidence of what it did.
3. **Every output got read** — including the boring ones. The dtype find, the sign convention, the flag-don't-delete call: all of them live in reading cells, not code.
4. **Nothing changed silently** — shapes before and after, a cleaning log with reasons.
5. **The chart served the decision**, was captioned in APA, and its blind spot was named.
6. **The AI use is logged**, with the rejection that proves a human was supervising.

**When would this become predictive?** The moment the supply-chain lead asks not *"where did we lose stock?"* but *"which product-branch pairs will exceed ₱500 shrinkage next month?"* — that is a prediction question with a target, a grain, and a horizon. With 23 events in 18 months, the honest answer today is: *not yet — collect more, or widen the event definition.* Knowing when the data cannot yet carry the model is not a failure of analytics. It **is** analytics.

*Now open your own slice, and hold your notebook to this standard.*
